# PPO Navigation — segway_1d_wheel

Adapted from Arthur's PPO notebook (Ozzy's segway_2).

**Same changes as the SAC adaptation:**

| | Arthur (segway_2) | This notebook |
|---|---|---|
| Model | segway_2.xml | **segway_1d_wheel.xml** |
| scipy | yes | **no — fast inline quaternion** |
| Ground Z | lookup table | **hardcoded 0.075** |
| ctrl | `ctrl[0]=-u, ctrl[1]=0` | **`ctrl[0]=-u` only** |
| shared_tracker | yes | **inline RunLogger** |
| Print every | 5 rollouts | **every rollout + every 10 ep** |


In [12]:
import mujoco, numpy as np, torch, torch.nn as nn, torch.optim as optim
import os, imageio, time, json
from collections import deque
from PIL import Image as PILImage, ImageDraw
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

XML_PATH = "segway_1d_wheel.xml"  # <-- update this

SAVE_DIR     = "SavedSeeds"
TRAIN_SEEDS  = [42]
VERIFY_SEEDS = [788, 999, 555, 321, 444]
os.makedirs(SAVE_DIR, exist_ok=True)

TORQUE_MAX   =  5.0    # wheel motor ctrlrange [-5, 5] Nm
GROUND_Z     =  0.075  # wheel radius — hardcoded, no lookup table
X_NORM       =  2.4
XD_NORM      =  5.0
TH_NORM      =  1.57
THD_NORM     =  5.0
X_DONE_LIMIT =  2.8
TH_DONE      =  0.5


## Inline RunLogger


In [13]:
class RunLogger:
    def __init__(self, algo, seed, log_every_steps=2000):
        self.algo=algo; self.seed=seed; self.log_every_steps=log_every_steps
        self.rows=[]; self._ep_rewards=[]; self._ep_arrives=[]; self._ep_losses=[]
        self._next_log=log_every_steps; self._t0=time.time()
    def episode_end(self, total_steps, ep_count, ep_ret, arrived, loss):
        self._ep_rewards.append(ep_ret); self._ep_arrives.append(float(arrived))
        self._ep_losses.append(loss)
        avg=float(np.mean(self._ep_rewards[-100:])); rp=float(np.mean(self._ep_arrives[-100:]))*100.
        if total_steps>=self._next_log:
            self.rows.append({"steps":total_steps,"avg_reward":avg,
                              "loss":float(np.mean(self._ep_losses[-100:])),"wall_time":time.time()-self._t0})
            self._next_log+=self.log_every_steps
        return avg, rp
    def save(self, save_dir, tag, eval_results=None):
        path=f"{save_dir}/{self.algo}_{tag}.json"
        with open(path,"w") as f:
            json.dump({"algo":self.algo,"seed":self.seed,"rows":self.rows,
                       "eval":eval_results,"total_wall_time":self.total_wall_time},f,indent=2)
        print(f"Log saved: {path}")
    @property
    def total_wall_time(self): return time.time()-self._t0


## Observation helpers

Same as SAC v3 — fast inline quaternion, no scipy.


In [14]:
def get_obs(data):
    """
    freejoint + wheel hinge:
      qpos = [x, y, z, qw, qx, qy, qz, wheel_angle]
      qvel = [vx, vy, vz, wx, wy, wz, wheel_speed]
    pitch = arcsin(2*(qw*qy - qz*qx))  — XYZ Tait-Bryan, valid for |theta|<90 deg
    """
    qw,qx,qy,qz = data.qpos[3],data.qpos[4],data.qpos[5],data.qpos[6]
    theta = float(np.arcsin(np.clip(2.0*(qw*qy - qz*qx), -1.0, 1.0)))
    return np.array([data.qpos[0], data.qvel[0], theta, data.qvel[4]], dtype=np.float32)

def normalize_obs(obs):
    x,xd,th,thd = obs
    return np.array([
        np.clip(x  /X_NORM,  -1,1),
        np.clip(xd /XD_NORM, -1,1),
        np.clip(th /TH_NORM, -1,1),
        np.clip(thd/THD_NORM,-1,1),
    ], dtype=np.float32)


## PPO Network and Reward

`NavPPO` is Arthur's exact architecture — unchanged.

- Shared backbone: `5 → 256 → 256 → 128` with Tanh
- Actor head: `mean` + learnable `log_std` parameter
- Critic head: single value output
- Actions squashed through tanh → `(-5, 5) Nm`


In [15]:
class NavPPO(nn.Module):
    def __init__(self, torque_max=TORQUE_MAX):
        super().__init__(); self.torque_max=torque_max
        self.net = nn.Sequential(
            nn.Linear(5,256), nn.Tanh(),
            nn.Linear(256,256), nn.Tanh(),
            nn.Linear(256,128), nn.Tanh(),
        )
        self.actor_mean    = nn.Linear(128,1)
        self.actor_log_std = nn.Parameter(torch.tensor([-0.5]))  # std~0.6
        self.critic        = nn.Linear(128,1)
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,0.5); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor_mean.weight,0.01)

    def forward(self, x):
        s=self.net(x); return self.actor_mean(s), self.critic(s)

    def get_action(self, obs, goal_x, deterministic=False):
        dist_norm=float(np.clip((goal_x-obs[0])/5.,-1,1))
        inp=torch.FloatTensor([*normalize_obs(obs),dist_norm])
        mean,val=self(inp); std=self.actor_log_std.exp().clamp(0.1,2.0)
        dist=torch.distributions.Normal(mean,std)
        raw=mean if deterministic else dist.rsample()
        logp=dist.log_prob(raw).sum(-1)
        torque=torch.tanh(raw)*self.torque_max
        return torque.item(), logp, val


def nav_reward(obs, prev_x, goal_x, at_goal, done, step):
    x,xd,theta,thd=obs
    if done:    return -20.0
    if at_goal: return  50.0
    r=(abs(prev_x-goal_x)-abs(x-goal_x))*15.0
    r-=abs(x-goal_x)*0.05
    r+=0.02
    if abs(theta)>0.20: r-=(abs(theta)-0.2)*5.0
    return float(r)


## PPO `train_nav`

Standard PPO with GAE. Key parameters (identical to Arthur):

| Param | Value |
|---|---|
| Rollout steps (STEPS) | 2048 |
| PPO epochs | 10 |
| Clip ε | 0.2 |
| GAE λ | 0.95 |
| Minibatch | 256 |
| Entropy coef | 0.01 → 0.002 over 3000 ep |

**Changes from Arthur:**
- `ctrl[0]=-u` only (no `ctrl[1]`)
- `GROUND_Z` hardcoded (no lookup)
- No scipy
- `max_steps` based (not `max_episodes`)
- Print every rollout + summary every 10 episodes


In [16]:
def train_nav(x_start=0.0, x_goal=2.0,
              seed=42, tag="nav_ppo_1d",
              max_steps=500_000):
    torch.manual_seed(seed); np.random.seed(seed)

    net       = NavPPO(torque_max=TORQUE_MAX)
    optimizer = optim.Adam(net.parameters(), lr=3e-4, eps=1e-5)

    STEPS        = 2048   # rollout length
    EPOCHS       = 10     # PPO update epochs per rollout
    CLIP         = 0.2
    GAMMA        = 0.99
    LAM          = 0.95
    MB           = 256    # minibatch size
    ENT_START    = 0.01
    ENT_END      = 0.002
    ENT_DECAY    = 3000   # episodes
    MAX_EP_STEPS = 3000

    rewards=[]; losses=[]; best_avg=-9999; best_weights=None
    recent_arrivals=deque(maxlen=200)

    def gae(rews, vals, dones, nv):
        adv=[]; g=0
        for r,v,d in zip(reversed(rews),reversed(vals),reversed(dones)):
            delta=r+GAMMA*nv*(1-d)-v
            g=delta+GAMMA*LAM*(1-d)*g
            adv.insert(0,g); nv=v
        return adv

    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)

    def reset_env():
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start
        data.qpos[2]=GROUND_Z          # wheel radius, no lookup table
        data.qpos[3:7]=[1,0,0,0]       # upright
        data.qvel[:]=0.0
        data.qvel[4]=np.random.uniform(-0.01,0.01)
        mujoco.mj_forward(model,data); return get_obs(data)

    def env_step(torque):
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
        data.ctrl[0]=-u           # same sign as Arthur; no ctrl[1]
        mujoco.mj_step(model,data)
        obs=get_obs(data); x,_,th,_=obs
        done=abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE
        at_goal=abs(x-x_goal)<0.10 and abs(th)<0.25
        return obs,done,at_goal

    def make_inp(obs):
        return np.array([*normalize_obs(obs),float(np.clip((x_goal-obs[0])/5.,-1,1))],dtype=np.float32)

    logger=RunLogger("PPO",seed,log_every_steps=2000)
    obs=reset_env(); ep_total=0.0; prev_x=obs[0]
    ep_count=ep_step=rollout=total_steps=0

    hdr="  Roll |    Ep |    Avg10 |   Avg100 |   Rec% |    Loss |   Std"
    print(f"\nPPO | A={x_start}m -> B={x_goal}m | budget={max_steps:,} steps")
    print(hdr); print("-"*len(hdr))

    recent_rewards=deque(maxlen=10)

    while total_steps < max_steps:
        # ── collect rollout ──────────────────────────────────────────────────
        ol,al,rl,vl,ll,dl=[],[],[],[],[],[]
        rollout_ep_rets=[]

        for _ in range(STEPS):
            with torch.no_grad():
                torque,logp,value=net.get_action(obs,x_goal,deterministic=False)
            ep_step+=1; total_steps+=1
            obs2,done,at_goal=env_step(torque)
            r=nav_reward(obs2,prev_x,x_goal,at_goal,done,ep_step)
            ep_total+=r
            ol.append(make_inp(obs)); al.append([torque])
            rl.append(r); vl.append(value.item())
            ll.append(logp.item()); dl.append(float(done or at_goal))
            prev_x=obs2[0]; obs=obs2
            if done or at_goal or ep_step>=MAX_EP_STEPS:
                recent_arrivals.append(float(at_goal))
                rewards.append(ep_total); recent_rewards.append(ep_total)
                rollout_ep_rets.append(ep_total)
                ep_count+=1; ep_total=0.0; ep_step=0
                obs=reset_env(); prev_x=obs[0]
            if total_steps>=max_steps: break

        # ── bootstrap value ──────────────────────────────────────────────────
        with torch.no_grad():
            _,nv=net(torch.FloatTensor(make_inp(obs)))
        adv=gae(rl,vl,dl,nv.item())
        ret=[a+v for a,v in zip(adv,vl)]

        # ── PPO update ───────────────────────────────────────────────────────
        obs_t =torch.FloatTensor(ol)
        act_t =torch.FloatTensor(al)
        old_lp=torch.FloatTensor(ll)
        adv_t =torch.FloatTensor(adv)
        ret_t =torch.FloatTensor(ret)
        adv_t =(adv_t-adv_t.mean())/(adv_t.std()+1e-8)  # normalise
        ENT=max(ENT_END, ENT_START-(ENT_START-ENT_END)*ep_count/ENT_DECAY)
        N=len(obs_t); batch_loss=[]

        for _ in range(EPOCHS):
            idx=torch.randperm(N)
            for start in range(0,N,MB):
                mb=idx[start:start+MB]
                mean,val=net(obs_t[mb])
                std=net.actor_log_std.exp().clamp(0.1,2.0)
                dist=torch.distributions.Normal(mean,std)
                # invert tanh to recover raw pre-squash action
                raw=torch.atanh(torch.clamp(act_t[mb]/net.torque_max,-0.9999,0.9999))
                new_lp=dist.log_prob(raw).sum(-1)
                ratio=(new_lp-old_lp[mb]).exp()
                mb_adv=adv_t[mb]
                obj=torch.min(ratio*mb_adv,ratio.clamp(1-CLIP,1+CLIP)*mb_adv)
                actor_loss=-obj.mean()
                value_loss=nn.MSELoss()(val.squeeze(),ret_t[mb])
                entropy=dist.entropy().mean()
                loss=actor_loss+0.5*value_loss-ENT*entropy
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(),0.5)
                optimizer.step(); batch_loss.append(loss.item())

        rollout+=1
        avg_loss=float(np.mean(batch_loss)); losses.append(avg_loss)
        avg100=float(np.mean(rewards[-100:])) if rewards else 0.0
        avg10 =float(np.mean(list(recent_rewards))) if recent_rewards else 0.0
        rec   =float(np.mean(list(recent_arrivals)))*100 if recent_arrivals else 0.0
        std_v =float(net.actor_log_std.exp().item())
        icon  ="OK" if avg100>0 else "UP" if avg100>-20 else ".."
        print(f"{rollout:>6} | {ep_count:>5} | {avg10:>8.2f} | {avg100:>8.2f} | "
              f"{rec:>5.1f}% | {avg_loss:>7.3f} | {std_v:>5.3f}  {icon}",flush=True)

        # log + checkpoint
        for ep_ret in rollout_ep_rets:
            arrived=recent_arrivals[-1] if recent_arrivals else False
            _,_=logger.episode_end(total_steps,ep_count,ep_ret,arrived,avg_loss)
        if avg100>best_avg:
            best_avg=avg100
            best_weights={k:v.clone() for k,v in net.state_dict().items()}
            torch.save(net.state_dict(),f"{SAVE_DIR}/nav_ppo_{tag}.pth")

    if best_weights: net.load_state_dict(best_weights)
    return net, rewards, losses, logger


## Train + Verify


In [17]:
def train_one_verify_many(algo_label, train_seed=42,
                          verify_seeds=(788,999,555,321,444),
                          x_start=0.0, x_goal=2.0, step_budget=500_000):
    verify_seeds=list(verify_seeds)
    print(f"\n{60*chr(61)}\n  {algo_label}  TRAIN seed={train_seed}\n{60*chr(61)}")
    net,rewards,losses,logger=train_nav(x_start=x_start,x_goal=x_goal,
        seed=train_seed,tag=f"cmp_seed{train_seed}",max_steps=step_budget)

    def eval_policy(net,seed,n=20):
        torch.manual_seed(seed); np.random.seed(seed)
        m=mujoco.MjModel.from_xml_path(XML_PATH); d=mujoco.MjData(m)
        strict=loose=fell=0; max_xs=[]; times=[]
        for _ in range(n):
            mujoco.mj_resetData(m,d)
            d.qpos[0]=x_start; d.qpos[2]=GROUND_Z
            d.qpos[3:7]=[1,0,0,0]; d.qvel[:]=0.0
            d.qvel[4]=np.random.uniform(-0.02,0.02)
            mujoco.mj_forward(m,d); obs=get_obs(d); mx=0.0; end="timeout"
            for step in range(3000):
                with torch.no_grad(): t,_,_=net.get_action(obs,x_goal,deterministic=True)
                d.ctrl[0]=-float(np.clip(t,-TORQUE_MAX,TORQUE_MAX))
                mujoco.mj_step(m,d); obs=get_obs(d); x,_,th,_=obs; mx=max(mx,x)
                if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE: end="fell"; break
                if abs(x-x_goal)<0.10 and abs(th)<0.35: end="strict"; times.append(step*m.opt.timestep); break
                elif abs(x-x_goal)<0.25 and abs(th)<0.35: end="loose"; times.append(step*m.opt.timestep)
            strict+=end=="strict"; loose+=end=="loose"; fell+=end=="fell"; max_xs.append(mx)
        return {"strict_pct":strict/n*100,"loose_pct":(strict+loose)/n*100,
                "fell_pct":fell/n*100,"avg_max_x":float(np.mean(max_xs)),
                "avg_time":float(np.mean(times)) if times else 999.0}

    print(f"\n  Verifying on {len(verify_seeds)} seeds...")
    print(f"  Seed   | Strict% | Loose% | Fell% | Time"); print("  "+"-"*44)
    per_seed={}
    for vs in verify_seeds:
        r=eval_policy(net,vs); per_seed[vs]=r
        icon="OK" if r["strict_pct"]>=80 else "~" if r["strict_pct"]>=50 else "X"
        print(f"  {vs:>6} | {r['strict_pct']:>6.0f}% | {r['loose_pct']:>5.0f}% | "
              f"{r['fell_pct']:>4.0f}% | {r['avg_time']:>5.1f}s  {icon}")
    summary={"train_seed":train_seed,"verify_seeds":verify_seeds,
             "strict_pct":float(np.mean([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "strict_std":float(np.std([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "fell_pct":float(np.mean([per_seed[v]["fell_pct"] for v in verify_seeds])),
             "avg_time":float(np.mean([per_seed[v]["avg_time"] for v in verify_seeds
                                        if per_seed[v]["avg_time"]<999] or [999])),"per_seed":per_seed}
    print(f"\n  avg: {summary['strict_pct']:.1f}% +/- {summary['strict_std']:.1f}% strict | {summary['fell_pct']:.1f}% fell")
    logger.save(SAVE_DIR,f"cmp_seed{train_seed}",eval_results=summary)
    return net,logger,summary


## Recording Function


In [18]:
def record_all_seeds(nav_net,train_seeds=[42],verify_seeds=[788,999,555,321,444],
                     x_start=0.0,x_goal=2.0,max_steps=3000):
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; renderer=mujoco.Renderer(model,height=480,width=640)
    cam=mujoco.MjvCamera(); cam.type=mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat=np.array([1.0,0.0,0.3]); cam.distance=4.5; cam.azimuth=90; cam.elevation=-15
    for seed,role in [(s,"TRAIN") for s in train_seeds]+[(s,"VERIFY") for s in verify_seeds]:
        torch.manual_seed(seed); np.random.seed(seed)
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start; data.qpos[2]=GROUND_Z
        data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0; mujoco.mj_forward(model,data)
        obs=get_obs(data); frames=[]; strict=arrived=fell=False
        for step in range(max_steps):
            with torch.no_grad(): torque,_,_=nav_net.get_action(obs,x_goal,deterministic=True)
            data.ctrl[0]=-float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
            mujoco.mj_step(model,data); obs=get_obs(data); x,_,theta,_=obs
            fell=abs(x)>X_DONE_LIMIT or abs(theta)>TH_DONE
            arrived=abs(x-x_goal)<0.25 and abs(theta)<0.35
            strict=abs(x-x_goal)<0.10 and abs(theta)<0.35
            renderer.update_scene(data,camera=cam)
            img=PILImage.fromarray(renderer.render()); draw=ImageDraw.Draw(img); W,H=img.size
            prog=float(np.clip(x/x_goal,0,1)); bw=W-40
            pcol=(0,200,0) if strict else (255,140,0) if arrived else (30,100,220)
            draw.rectangle([20,8,W-20,28],fill=(40,40,40))
            draw.rectangle([20,8,20+int(bw*prog),28],fill=pcol)
            draw.text((22,10),"A",fill=(255,255,255)); draw.text((W-28,10),"B",fill=(255,255,255))
            draw.text((W//2-40,10),f"{prog*100:.0f}%  x={x:.3f}m",fill=(255,255,255))
            badge_col=(0,60,140) if role=="TRAIN" else (100,0,140)
            draw.rectangle([8,34,170,58],fill=badge_col)
            draw.text((12,38),f"[{role}] seed={seed}",fill=(255,255,255))
            if strict:    sc,st=(0,120,0),  f"STRICT x={x:.3f}m t={step*dt:.1f}s"
            elif arrived: sc,st=(120,100,0),f"LOOSE  x={x:.3f}m t={step*dt:.1f}s"
            elif fell:    sc,st=(140,0,0),  f"FELL   x={x:.3f}m th={np.degrees(theta):.1f}deg"
            else:         sc,st=(20,20,70), f"x={x:+.3f}m th={np.degrees(theta):+.1f}deg u={torque:+.2f}Nm t={step*dt:.1f}s"
            draw.rectangle([175,34,W-8,58],fill=sc); draw.text((178,38),st,fill=(255,255,255))
            frames.append(np.array(img))
            if fell or strict: break
        fname=f"{SAVE_DIR}/ppo_{role.lower()}_seed{seed}.gif"
        imageio.mimsave(fname,frames,fps=30)
        status="STRICT" if strict else "LOOSE" if arrived else "FELL" if fell else "TIMEOUT"
        print(f"  [{role}] seed={seed} -> {status}  {fname}")
    print("Done.")


## Plot Loss and Reward


In [19]:
def plot_loss_reward(logger=None,rewards=None,losses=None,algo="model",save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    if logger is not None:
        sx=[r["steps"] for r in logger.rows]; ry=[r["avg_reward"] for r in logger.rows]
        ly=[r["loss"] for r in logger.rows]; wt=getattr(logger,"total_wall_time",None); hs=True
    else:
        hs=False
    fig,(axr,axl)=plt.subplots(1,2,figsize=(15,5))
    title=f"{algo} Training"
    if hs and wt: title+=f"  (wall: {wt/60:.1f} min)"
    fig.suptitle(title,fontsize=13,fontweight="bold")
    if hs:
        axr.plot(sx,ry,color="#E74C3C",lw=2.5,label=f"avg100 (peak={max(ry):.1f})")
        axl.plot(sx,ly,color="#9B59B6",lw=2,label="PPO Loss")
        axr.set_xlabel("Steps"); axl.set_xlabel("Steps")
    else:
        r=np.array(rewards); w=min(100,max(2,len(r)//10))
        axr.plot(np.convolve(r,np.ones(w)/w,mode="valid"),color="#E74C3C",lw=2)
        l=np.array(losses); axl.plot(l,color="#9B59B6",alpha=0.5,lw=1)
        axr.set_xlabel("Episode"); axl.set_xlabel("Rollout")
    axr.axhline(0,color="green",ls="--",alpha=0.5); axr.set_title("Reward",fontweight="bold"); axr.grid(alpha=0.3)
    axl.axhline(0,color="green",ls="--",alpha=0.5); axl.set_title("PPO Loss",fontweight="bold"); axl.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_training.png",dpi=150,bbox_inches="tight"); plt.show()
    if hs and wt: print(f"Wall time: {wt/60:.1f} min")


## Plot Episode Traces


In [10]:
def plot_episode_traces(net,algo="model",x_goal=2.0,save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; mujoco.mj_resetData(model,data)
    data.qpos[0]=0.0; data.qpos[2]=GROUND_Z; data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0
    mujoco.mj_forward(model,data); obs=get_obs(data)
    ts,xs,ths,torqs=[],[],[],[]
    for step in range(3000):
        with torch.no_grad(): torque,_,_=net.get_action(obs,x_goal,deterministic=True)
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX)); data.ctrl[0]=-u
        mujoco.mj_step(model,data); obs=get_obs(data); x,_,th,_=obs
        ts.append(step*dt); xs.append(x); ths.append(np.degrees(th)); torqs.append(torque)
        if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE or (abs(x-x_goal)<0.25 and abs(th)<0.35): break
    ts,xs,ths,torqs=np.array(ts),np.array(xs),np.array(ths),np.array(torqs)
    arrived=abs(xs[-1]-x_goal)<0.25 and abs(ths[-1])<20
    rms=np.sqrt(np.mean(torqs**2))
    fig=plt.figure(figsize=(15,9)); gs=gridspec.GridSpec(2,2,hspace=0.38,wspace=0.25)
    fig.suptitle(f"{algo} Episode",fontsize=13,fontweight="bold")
    ax=fig.add_subplot(gs[0,:]); ax.plot(ts,xs,"#E74C3C",lw=2.5,label="x position")
    ax.axhline(0,color="blue",ls=":",lw=2); ax.axhline(x_goal,color="green",ls=":",lw=2,label=f"goal {x_goal}m")
    ax.axhspan(x_goal-0.25,x_goal+0.25,alpha=0.1,color="green",label="Goal zone")
    if arrived:
        idx=np.where(np.abs(xs-x_goal)<0.25)[0][0]
        ax.axvline(ts[idx],color="green",ls="--",alpha=0.6)
        ax.annotate(f"ARRIVED t={ts[idx]:.1f}s",xy=(ts[idx],xs[idx]),
                    xytext=(ts[idx]+0.3,x_goal-0.4),color="green",fontsize=9,
                    arrowprops=dict(arrowstyle="->",color="green"))
    ax.set_xlabel("Time(s)"); ax.set_ylabel("x(m)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,0]); ax.plot(ts,ths,"#E74C3C",lw=2,label="theta (deg)")
    ax.axhspan(-20,20,alpha=0.06,color="green",label="Stable +-20 deg")
    ax.axhline(0,color="gray",alpha=0.4)
    ax.set_xlabel("Time(s)"); ax.set_ylabel("Tilt(deg)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,1]); ax.plot(ts,torqs,"#2ECC71",lw=2,label=f"Torque (RMS={rms:.2f}Nm)")
    ax.fill_between(ts,torqs,0,where=(torqs>0),alpha=0.15,color="red",label="Forward")
    ax.fill_between(ts,torqs,0,where=(torqs<0),alpha=0.15,color="blue",label="Backward")
    ax.axhline(TORQUE_MAX,color="orange",ls=":",lw=1.5,label=f"+-{TORQUE_MAX}Nm")
    ax.axhline(-TORQUE_MAX,color="orange",ls=":",lw=1.5)
    ax.axhline(0,color="gray",alpha=0.4)
    ax.set_xlabel("Time(s)"); ax.set_ylabel("Torque(Nm)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.savefig(f"{save_prefix}_traces.png",dpi=150,bbox_inches="tight"); plt.show()
    print(f"  Duration {ts[-1]:.1f}s  MaxX {max(xs):.3f}m  AvgTilt {np.mean(np.abs(ths)):.1f}deg  RMS {rms:.3f}Nm")


## RUN ALL

### At 500,000 Steps


In [10]:
net, logger, summary = train_one_verify_many("PPO", train_seed=42)



  PPO  TRAIN seed=42

PPO | A=0.0m -> B=2.0m | budget=500,000 steps
  Roll |    Ep |    Avg10 |   Avg100 |   Rec% |    Loss |   Std
---------------------------------------------------------------


C:\Users\edward\AppData\Local\Temp\ipykernel_21120\4024654379.py:95: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  obs_t =torch.FloatTensor(ol)


     1 |    14 |   -63.34 |   -63.90 |   0.0% |  39.714 | 0.606  ..
     2 |    30 |   -57.92 |   -60.36 |   0.0% |  23.335 | 0.604  ..
     3 |    42 |   -68.18 |   -61.70 |   0.0% |  15.547 | 0.602  ..
     4 |    56 |   -65.46 |   -63.24 |   0.0% |  15.435 | 0.601  ..
     5 |    72 |   -58.82 |   -61.89 |   0.0% |  10.760 | 0.597  ..
     6 |    86 |   -58.49 |   -61.55 |   0.0% |   8.400 | 0.587  ..
     7 |   102 |   -54.75 |   -60.82 |   0.0% |   8.709 | 0.585  ..
     8 |   125 |   -50.28 |   -58.75 |   0.0% |   5.810 | 0.587  ..
     9 |   146 |   -50.25 |   -55.70 |   0.0% |   9.910 | 0.586  ..
    10 |   161 |   -56.15 |   -55.40 |   0.0% |   6.782 | 0.584  ..
    11 |   174 |   -64.93 |   -56.06 |   0.0% |  11.066 | 0.584  ..
    12 |   191 |   -53.43 |   -55.38 |   0.0% |   4.634 | 0.582  ..
    13 |   211 |   -50.64 |   -54.64 |   0.0% |   3.516 | 0.585  ..
    14 |   231 |   -47.05 |   -54.24 |   0.0% |  11.749 | 0.587  ..
    15 |   251 |   -50.99 |   -53.36 |   0.0% | 

In [ ]:
record_all_seeds(net, train_seeds=[42], verify_seeds=[788,999,555], x_goal=2.0)


  [TRAIN] seed=42 -> FELL  SavedSeeds/ppo_train_seed42.gif
  [VERIFY] seed=788 -> FELL  SavedSeeds/ppo_verify_seed788.gif
  [VERIFY] seed=999 -> FELL  SavedSeeds/ppo_verify_seed999.gif
  [VERIFY] seed=555 -> FELL  SavedSeeds/ppo_verify_seed555.gif
Done.


: 

In [ ]:
plot_loss_reward(logger=logger, algo="PPO")


In [ ]:
plot_episode_traces(net, algo="PPO")


### At 3,000,000 Steps


In [20]:
net, logger, summary = train_one_verify_many("PPO_1M", train_seed=42, step_budget=3_000_000)



  PPO_1M  TRAIN seed=42

PPO | A=0.0m -> B=2.0m | budget=3,000,000 steps
  Roll |    Ep |    Avg10 |   Avg100 |   Rec% |    Loss |   Std
---------------------------------------------------------------
     1 |    14 |   -63.34 |   -63.90 |   0.0% |  39.714 | 0.606  ..
     2 |    30 |   -57.92 |   -60.36 |   0.0% |  23.335 | 0.604  ..
     3 |    42 |   -68.18 |   -61.70 |   0.0% |  15.547 | 0.602  ..
     4 |    56 |   -65.46 |   -63.24 |   0.0% |  15.435 | 0.601  ..
     5 |    72 |   -58.82 |   -61.89 |   0.0% |  10.760 | 0.597  ..
     6 |    86 |   -58.49 |   -61.55 |   0.0% |   8.400 | 0.587  ..
     7 |   102 |   -54.75 |   -60.82 |   0.0% |   8.709 | 0.585  ..
     8 |   125 |   -50.28 |   -58.75 |   0.0% |   5.810 | 0.587  ..
     9 |   146 |   -50.25 |   -55.70 |   0.0% |   9.910 | 0.586  ..
    10 |   161 |   -56.15 |   -55.40 |   0.0% |   6.782 | 0.584  ..
    11 |   174 |   -64.93 |   -56.06 |   0.0% |  11.066 | 0.584  ..
    12 |   191 |   -53.43 |   -55.38 |   0.0% |   

In [21]:
record_all_seeds(net, train_seeds=[42], verify_seeds=[788,999,555], x_goal=2.0)

  [TRAIN] seed=42 -> STRICT  SavedSeeds/ppo_train_seed42.gif
  [VERIFY] seed=788 -> STRICT  SavedSeeds/ppo_verify_seed788.gif
  [VERIFY] seed=999 -> STRICT  SavedSeeds/ppo_verify_seed999.gif
  [VERIFY] seed=555 -> STRICT  SavedSeeds/ppo_verify_seed555.gif
Done.
